Модули

In [ ]:
import os
import requests
import json
import Settings

Написание запроса

#TODO 

Ошибка в написании url. Данный запрос будет выводить тебе только один фильм. И согласно документации 

https://kinopoiskdev.readme.io/reference/moviecontroller_getrandommoviev1_4

конкретно в этом запросе нет возможности выбрать сколько фильмов выводить. поэтому и выводится единичный фильм. 

Обрати внимание на файл Create_BD.ipynb. В первом блоке есть одна ссылка, по которой получена начальная БД.  
Попробуй использовать псевдо поиск. В нём можно получить до 250 фильмов за раз.

https://kinopoiskdev.readme.io/reference/moviecontroller_findmanybyqueryv1_4


In [ ]:
# URL для запроса к API
url = "https://api.kinopoisk.dev/v1.4/movie/random"

# Заголовки для запроса
headers = {
    "accept": "application/json",
    "X-API-KEY": Settings.XAPIKEY
}


In [ ]:

formatted_data=[]
response = requests.get(url, headers=headers)
formatted_data_list=[]

existing_data = []
if os.path.exists('all_film_list.json'):
    with open('all_film_list.json', 'r') as file:
        existing_data = json.load(file)

for _ in range(100):
    response = requests.get(url, headers=headers)
    if response.status_code == 200:
        response_json = response.json()
# # Проверка успешности запроса
# if response.status_code == 200:
#     response_json = response.json()

        # Функция для преобразования данных в нужный формат
        def format_movie_data(movie_data):
            if not movie_data.get("description"):
                return None
            if not movie_data.get("alternativeName") and not movie_data.get("name"):
                return None
            if not movie_data.get("description") and not movie_data.get("shortDescription") and \
                not movie_data.get("slogan") and not movie_data.get("poster", {}).get("url"):
                return None
            if not movie_data.get("slogan") and not movie_data.get("poster", {}).get("url"):
                return None
            if not movie_data.get("poster", {}).get("url"):
                return None

            formatted_data = {
                "id": movie_data.get("id"),
                "externalId": movie_data.get("externalId", {}),
                "name": movie_data.get("name"),
                "alternativeName": movie_data.get("alternativeName"),
                "enName": movie_data.get("enName"),
                "year": movie_data.get("year"),
                "description": movie_data.get("description"),
                "shortDescription": movie_data.get("shortDescription"),
                "slogan": movie_data.get("slogan"),
                "rating": movie_data.get("rating", {}),
                "votes": movie_data.get("votes", {}),
                "movieLength": movie_data.get("movieLength"),
                "poster": movie_data.get("poster", {}),
                "backdrop": movie_data.get("backdrop", {}),
                "genres": [{"name": genre["name"]} for genre in movie_data.get("genres", [])],
                "countries": [{"name": country["name"]} for country in movie_data.get("countries", [])],
                "premiere": movie_data.get("premiere", {}),
                "ticketsOnSale": movie_data.get("ticketsOnSale"),
                "sequelsAndPrequels": [
                    {
                        "id": sequel["id"],
                        "alternativeName": sequel["alternativeName"],
                        "year": sequel["year"],
                        "rating": sequel["rating"]
                    }
                    for sequel in movie_data.get("sequelsAndPrequels", [])
                ],
                "persons": [
                    {
                        "id": person["id"],
                        "name": person["name"],
                        "enName": person["enName"],
                        "description": person["description"],
                        "profession": person["profession"],
                        "enProfession": person["enProfession"],
                        "photo": person["photo"]
                    }
                    for person in movie_data.get("persons", [])
                ]
            }
            
            return formatted_data

        # Преобразование данных в нужный формат
        formatted_data = format_movie_data(response_json)
        if formatted_data != None:
            # Вывод данных в нужном формате
            print(json.dumps({"docs": [formatted_data]}, indent=4, ensure_ascii=False))
            #response_json = json.dumps({"docs": [formatted_data]}, indent=4, ensure_ascii=False)
            # with open('all_film_list.json', 'w') as file:
            #     # Записываем JSON-строку в файл
            #     file.write(response_json)
            

            formatted_data_list.append(formatted_data)  
            # Fetch and process new movie data
            # Чтение существующих данных из файла

            # Объединение существующих данных с новыми данными
            existing_data.extend(formatted_data_list)

            # Сохранение обновленных данных в файл
            with open('all_film_list.json', 'w') as file:
                json.dump(existing_data, file, indent=4, ensure_ascii=False)

    else:   
        print(f"Ошибка запроса: {response.status_code}")



